In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, roc_auc_score)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier,  GradientBoostingClassifier, VotingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier


In [2]:
data=pd.read_csv('loan_prediction.csv')

In [30]:
data.head()

,Gender,Married,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Loan_Status,Dependents_0,Dependents_1,Dependents_2,Dependents_3+,Property_Area_Rural,Property_Area_Semiurban,Property_Area_Urban
0,1,0,1,0,0.072991,-0.554487,0.000000,0.279851,1,1,1,0,0,0,0,0,1
1,1,1,1,0,-0.134412,-0.038732,-0.219273,0.279851,1,0,0,1,0,0,1,0,0
2,1,1,1,1,-0.393747,-0.554487,-0.957641,0.279851,1,1,1,0,0,0,0,0,1
3,1,1,0,0,-0.462062,0.251980,-0.314547,0.279851,1,1,1,0,0,0,0,0,1
4,1,0,1,0,0.097728,-0.554487,-0.064454,0.279851,1,1,1,0,0,0,0,0,1


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    object 
 1   Gender             601 non-null    object 
 2   Married            611 non-null    object 
 3   Dependents         599 non-null    object 
 4   Education          614 non-null    object 
 5   Self_Employed      582 non-null    object 
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    object 
 12  Loan_Status        614 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 62.5+ KB


In [5]:
data.columns

Index(['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education',
       'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status'],
      dtype='object')

In [31]:
for col in data.select_dtypes('object').columns:
  print(f'{col} ------> {data[col].unique()}')

In [7]:
from sklearn.impute import SimpleImputer


In [8]:
cat_cols = data.select_dtypes(include='object').columns

In [9]:
num_cols=data.select_dtypes(include=['int64', 'float64']).columns

In [10]:
imp=SimpleImputer(strategy='most_frequent')
data[cat_cols]=imp.fit_transform(data[cat_cols])

In [11]:
data['Credit_History']=data['Credit_History'].replace(np.nan, 1)

In [12]:
inp=SimpleImputer(strategy='mean')
data[num_cols]=inp.fit_transform(data[num_cols])

In [13]:
data['Credit_History'] = data['Credit_History'].astype('int64')

In [14]:
data['Dependents'].value_counts()

Dependents
0     360
1     102
2     101
3+     51
Name: count, dtype: int64

In [15]:
data.drop('Loan_ID', axis=1, inplace=True)

In [16]:
data['Gender'] = data['Gender'].replace({'Male': 1, 'Female': 0})

C:\Users\aayse\AppData\Local\Temp\ipykernel_4440\924206060.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['Gender'] = data['Gender'].replace({'Male': 1, 'Female': 0})


In [17]:
data['Married'] = data['Married'].replace({'Yes': 1, 'No': 0})

C:\Users\aayse\AppData\Local\Temp\ipykernel_4440\673094510.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['Married'] = data['Married'].replace({'Yes': 1, 'No': 0})


In [18]:
data['Education'] = data['Education'].replace({'Graduate': 1, 'Not Graduate': 0})

C:\Users\aayse\AppData\Local\Temp\ipykernel_4440\982967041.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['Education'] = data['Education'].replace({'Graduate': 1, 'Not Graduate': 0})


In [19]:
data['Self_Employed'] = data['Self_Employed'].replace({'Yes': 1, 'No': 0})

C:\Users\aayse\AppData\Local\Temp\ipykernel_4440\642483772.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['Self_Employed'] = data['Self_Employed'].replace({'Yes': 1, 'No': 0})


In [20]:
data['Loan_Status'] = data['Loan_Status'].replace({'Y': 1, 'N': 0})

C:\Users\aayse\AppData\Local\Temp\ipykernel_4440\2166052920.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['Loan_Status'] = data['Loan_Status'].replace({'Y': 1, 'N': 0})


In [21]:
dependents_dummies = pd.get_dummies(data['Dependents'], prefix='Dependents', drop_first=False, dtype=int)

In [22]:
sc_columns = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term']

In [23]:
data = pd.concat([data, dependents_dummies], axis=1)

In [24]:
data.drop('Dependents', axis=1, inplace=True)

In [25]:
pro_dummies = pd.get_dummies(data['Property_Area'], prefix='Property_Area', drop_first=False, dtype=int)

In [26]:
data = pd.concat([data, pro_dummies], axis=1)

In [27]:
data.drop('Property_Area', axis=1, inplace=True)

In [28]:
sc=StandardScaler()

In [29]:
data[sc_columns] = sc.fit_transform(data[sc_columns])

In [32]:
x=data.drop(['Loan_Status'], axis=1)
y=data['Loan_Status']

In [33]:
x_train,x_test, y_train, y_test=train_test_split(x,y, test_size=0.2, random_state=42, stratify=y, shuffle=True)

In [38]:
dt=DecisionTreeClassifier( max_depth=10,
                           min_samples_leaf=6,
                           class_weight='balanced',
                           random_state=42)

In [39]:
bagging=BaggingClassifier(estimator=dt,
                          n_estimators=100,
                          random_state=42,
                          n_jobs=-1)


In [40]:
bagging.fit(x_train, y_train)
y_pred_bag=bagging.predict(x_test)
y_proba_bag=bagging.predict_proba(x_test)[:,1]
print("Bagging:")
print(accuracy_score(y_test, y_pred_bag))
print(confusion_matrix(y_test, y_pred_bag))
print(classification_report(y_test, y_pred_bag))
print(roc_auc_score(y_test, y_proba_bag))

Bagging:
0.8292682926829268
[[25 13]
 [ 8 77]]
              precision    recall  f1-score   support

           0       0.76      0.66      0.70        38
           1       0.86      0.91      0.88        85

    accuracy                           0.83       123
   macro avg       0.81      0.78      0.79       123
weighted avg       0.83      0.83      0.83       123

0.7989164086687307


In [42]:
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

gb_model.fit(x_train, y_train)

,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [43]:
y_pred_gb = gb_model.predict(x_test)
y_proba_gb = gb_model.predict_proba(x_test)[:, 1]

print("Gradient Boosting:")
print(accuracy_score(y_test, y_pred_gb))
print(confusion_matrix(y_test, y_pred_gb))
print(classification_report(y_test, y_pred_gb))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_gb))

Gradient Boosting:
0.8292682926829268
[[22 16]
 [ 5 80]]
              precision    recall  f1-score   support

           0       0.81      0.58      0.68        38
           1       0.83      0.94      0.88        85

    accuracy                           0.83       123
   macro avg       0.82      0.76      0.78       123
weighted avg       0.83      0.83      0.82       123

ROC-AUC: 0.7678018575851393


In [44]:
xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

xgb_model.fit(x_train, y_train)

C:\Users\aayse\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [21:29:31] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [45]:
y_pred_xgb = xgb_model.predict(x_test)
y_proba_xgb = xgb_model.predict_proba(x_test)[:, 1]

print("XGBoost:")
print(accuracy_score(y_test, y_pred_xgb))
print(confusion_matrix(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_xgb))

XGBoost:
0.7967479674796748
[[23 15]
 [10 75]]
              precision    recall  f1-score   support

           0       0.70      0.61      0.65        38
           1       0.83      0.88      0.86        85

    accuracy                           0.80       123
   macro avg       0.77      0.74      0.75       123
weighted avg       0.79      0.80      0.79       123

ROC-AUC: 0.7743034055727555


In [48]:
lgbm_model = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=-1,
    random_state=42
)

lgbm_model.fit(x_train, y_train)

[LightGBM] [Info] Number of positive: 337, number of negative: 154
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000203 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 380
[LightGBM] [Info] Number of data points in the train set: 491, number of used features: 16
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.686354 -> initscore=0.783130
[LightGBM] [Info] Start training from score 0.783130
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [49]:
y_pred_lgbm = lgbm_model.predict(x_test)
y_proba_lgbm = lgbm_model.predict_proba(x_test)[:, 1]

print("LGBM:")
print(accuracy_score(y_test, y_pred_lgbm))
print(confusion_matrix(y_test, y_pred_lgbm))
print(classification_report(y_test, y_pred_lgbm))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_lgbm))

LGBM:
0.8211382113821138
[[25 13]
 [ 9 76]]
              precision    recall  f1-score   support

           0       0.74      0.66      0.69        38
           1       0.85      0.89      0.87        85

    accuracy                           0.82       123
   macro avg       0.79      0.78      0.78       123
weighted avg       0.82      0.82      0.82       123

ROC-AUC: 0.7832817337461301


In [50]:
cat = CatBoostClassifier(verbose=0)
cat.fit(x_train, y_train)

y_pred_cat = cat.predict(x_test)
y_prob_cat = cat.predict_proba(x_test)[:,1]

print("LGBM:")
print(accuracy_score(y_test, y_pred_cat))
print(confusion_matrix(y_test, y_pred_cat))
print(classification_report(y_test, y_pred_cat))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_cat))

LGBM:
0.8292682926829268
[[23 15]
 [ 6 79]]
              precision    recall  f1-score   support

           0       0.79      0.61      0.69        38
           1       0.84      0.93      0.88        85

    accuracy                           0.83       123
   macro avg       0.82      0.77      0.78       123
weighted avg       0.83      0.83      0.82       123

ROC-AUC: 0.7969040247678019


In [52]:
voting = VotingClassifier(
    estimators=[
        ('gb', gb_model),
        ('xgb', xgb_model),
        ('lgb', lgbm_model)
    ],
    voting='soft'
)

voting.fit(x_train, y_train)
y_pred_vote = voting.predict(x_test)
y_prob_vote = voting.predict_proba(x_test)[:,1]

print("voting:")
print(accuracy_score(y_test, y_pred_vote))
print(confusion_matrix(y_test, y_pred_vote))
print(classification_report(y_test, y_pred_vote))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_vote))

C:\Users\aayse\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [21:35:54] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[LightGBM] [Info] Number of positive: 337, number of negative: 154
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000275 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 380
[LightGBM] [Info] Number of data points in the train set: 491, number of used features: 16
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.686354 -> initscore=0.783130
[LightGBM] [Info] Start training from score 0.783130
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[